# Lesson 6.2 — Lookahead and What-If

Run several candidate futures from the current state and compare. Reproducible under a fixed seed.

In [1]:
import numpy as np
from modules.module09.integration import (GreenhouseWorld, model_perception, understand,
                                          run_pipeline)
from modules.module10.twin import (DigitalTwin, GroundTruth, outcome_gap,
                                   monitor, predict, compare_futures)
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
ground = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1))
twin = DigitalTwin(ground.world); twin.sync(ground.report())
scen = {'nominal': None, 'obstacle': {v: dict(obstacle=(vxy,0.25))}}
futures = compare_futures(twin, scen, rng_seed=7)
print('nominal harvests %s: %s | obstacle skips %s'
      % (v, v in futures['nominal']['harvested'], futures['obstacle']['skipped']))
checks.append(v in futures['nominal']['harvested'])           # nominal harvests v
checks.append(v in futures['obstacle']['skipped'])            # obstacle future skips v
# the comparison localises the cost of the obstacle to v
g = outcome_gap(futures['nominal'], futures['obstacle'])
checks.append(v in g['skipped_only_in_real'] or v in g['harvested_only_in_sim'])
# reproducible under same seed
again = compare_futures(twin, scen, rng_seed=7)
checks.append(again['obstacle']['skipped'] == futures['obstacle']['skipped'])
print('cost of obstacle localised to %s; futures reproducible: OK' % v)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


nominal harvests F3: True | obstacle skips ['F3']


cost of obstacle localised to F3; futures reproducible: OK
4/4 checks passed.
All checks passed.
